# Pocket Screener Backtesting

Offline notebook untuk menghitung snapshot backtesting BSJP dan BPJS dari `data/historical_daily.csv`, lalu export hasil ke `public/backtesting/`.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CSV_PATH = ROOT / 'data' / 'historical_daily.csv'
OUT_DIR = ROOT / 'public' / 'backtesting'
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CSV_PATH, parse_dates=['date'])
df = df.sort_values(['symbol', 'date']).reset_index(drop=True)
df.head()

,symbol,date,open,high,low,close,volume
0,ADRO.JK,2021-05-17,1210.0,1215.0,1180.0,1185.0,69060200
1,ADRO.JK,2021-05-18,1200.0,1200.0,1170.0,1200.0,31413900
2,ADRO.JK,2021-05-19,1195.0,1195.0,1175.0,1175.0,23994100
3,ADRO.JK,2021-05-20,1175.0,1185.0,1170.0,1170.0,29580300
4,ADRO.JK,2021-05-21,1190.0,1195.0,1160.0,1165.0,45227600


In [2]:
MIN_DAILY_VALUE = 5_000_000_000

def add_features(group: pd.DataFrame) -> pd.DataFrame:
    group = group.copy()
    group['previous_close'] = group['close'].shift(1)
    group['previous_volume'] = group['volume'].shift(1)
    group['ma5'] = group['close'].rolling(5).mean()
    group['value'] = group['close'] * group['volume']
    group['next_open'] = group['open'].shift(-1)
    return group

features = pd.concat([add_features(group) for _, group in df.groupby('symbol', sort=False)], ignore_index=True)

features['bsjp_signal'] = (
    (features['close'] >= 1.05 * features['previous_close']) &
    (features['close'] >= features['ma5']) &
    (features['volume'] >= 1.2 * features['previous_volume']) &
    (features['value'] > MIN_DAILY_VALUE)
)

features['bpjs_signal'] = (
    (features['close'] >= features['ma5']) &
    (features['close'] >= 1.05 * features['previous_close']) &
    (features['close'] >= features['open']) &
    (features['volume'] >= 0.2 * features['previous_volume']) &
    (features['value'] > MIN_DAILY_VALUE)
)

features[['symbol', 'date', 'close', 'volume', 'bsjp_signal', 'bpjs_signal']].tail()

,symbol,date,close,volume,bsjp_signal,bpjs_signal
18025,UNVR.JK,2026-05-11,1785.0,15722700,False,False
18026,UNVR.JK,2026-05-12,1800.0,15541900,False,False
18027,UNVR.JK,2026-05-13,1785.0,6834600,False,False
18028,UNVR.JK,2026-05-14,1785.0,0,False,False
18029,UNVR.JK,2026-05-15,1785.0,0,False,False


In [3]:
def max_drawdown(returns: pd.Series):
    if returns.empty:
        return None
    equity = (1 + returns).cumprod()
    drawdown = equity / equity.cummax() - 1
    return float(drawdown.min())

def summarize(strategy: str, trades: pd.DataFrame):
    returns = trades['return_pct'] if not trades.empty else pd.Series(dtype=float)
    cumulative_return = None if returns.empty else float((1 + returns).prod() - 1)
    winrate = None if trades.empty else float((trades['return_pct'] > 0).mean())
    average_return = None if returns.empty else float(returns.mean())
    return {
        'strategy': strategy,
        'generatedAt': pd.Timestamp.utcnow().isoformat(),
        'source': 'data/historical_daily.csv',
        'symbols': sorted(features['symbol'].dropna().unique().tolist()),
        'periodStart': features['date'].min().date().isoformat(),
        'periodEnd': features['date'].max().date().isoformat(),
        'totalTrades': int(len(trades)),
        'winrate': winrate,
        'cumulativeReturn': cumulative_return,
        'averageReturn': average_return,
        'maxDrawdown': max_drawdown(returns),
        'sampleTrades': trades.tail(50).to_dict(orient='records'),
    }

bsjp_trades = features[features['bsjp_signal'] & features['next_open'].notna()].copy()
bsjp_trades['entry'] = bsjp_trades['close']
bsjp_trades['exit'] = bsjp_trades['next_open']
bsjp_trades['return_pct'] = bsjp_trades['exit'] / bsjp_trades['entry'] - 1

features['next_close'] = features.groupby('symbol')['close'].shift(-1)

bpjs_trades = features[features['bpjs_signal'] & features['next_open'].notna()].copy()
bpjs_trades['entry'] = bpjs_trades['next_open']
bpjs_trades['exit'] = bpjs_trades['next_close']
bpjs_trades['return_pct'] = bpjs_trades['exit'] / bpjs_trades['entry'] - 1

bsjp_result = summarize('BSJP', bsjp_trades[['symbol', 'date', 'entry', 'exit', 'return_pct']])
bpjs_result = summarize('BPJS', bpjs_trades[['symbol', 'date', 'entry', 'exit', 'return_pct']])
bsjp_result, bpjs_result

({'strategy': 'BSJP',
  'generatedAt': '2026-05-30T13:00:43.985081+00:00',
  'source': 'data/historical_daily.csv',
  'symbols': ['ADRO.JK',
   'ANTM.JK',
   'ASII.JK',
   'BBCA.JK',
   'BBRI.JK',
   'BMRI.JK',
   'BRIS.JK',
   'CPIN.JK',
   'ICBP.JK',
   'INCO.JK',
   'INDF.JK',
   'MDKA.JK',
   'PGAS.JK',
   'TLKM.JK',
   'UNVR.JK'],
  'periodStart': '2021-05-17',
  'periodEnd': '2026-05-15',
  'totalTrades': 361,
  'winrate': 0.5623268698060941,
  'cumulativeReturn': 3.8544200116672416,
  'averageReturn': 0.004474425004860104,
  'maxDrawdown': -0.13002547681012444,
  'sampleTrades': [{'symbol': 'PGAS.JK',
    'date': Timestamp('2024-04-30 00:00:00'),
    'entry': 1470.0,
    'exit': 1445.0,
    'return_pct': -0.017006802721088454},
   {'symbol': 'PGAS.JK',
    'date': Timestamp('2024-06-21 00:00:00'),
    'entry': 1505.0,
    'exit': 1505.0,
    'return_pct': 0.0},
   {'symbol': 'PGAS.JK',
    'date': Timestamp('2024-11-25 00:00:00'),
    'entry': 1570.0,
    'exit': 1570.0,
    're

In [4]:
with open(OUT_DIR / 'bsjp_result.json', 'w', encoding='utf-8') as f:
    json.dump(bsjp_result, f, indent=2, default=str)

with open(OUT_DIR / 'bpjs_result.json', 'w', encoding='utf-8') as f:
    json.dump(bpjs_result, f, indent=2, default=str)

OUT_DIR

WindowsPath('e:/Tugas Kuliah/Project/Screener & Prediction/pocket-screener/public/backtesting')